In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable


In [0]:
%run /Workspace/consolidated_pipeline/setup/utilities




In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sports-bar-de/{data_source}'
landing_path = f'{base_path}/landing'
processed_path = f'{base_path}/processed'
print(f'base_path: {base_path}')
print(f'landing_path: {landing_path}')
print(f'processed_path: {processed_path}')

bronze_table = f'{catalog}.{bronze_schema}.{data_source}'
silver_table = f'{catalog}.{silver_schema}.{data_source}'
gold_table = f'{catalog}.{gold_schema}.{data_source}'

In [0]:
df = spark.read.options(header=True,inferscema=True).csv(f'{processed_path}/*.csv').withColumn('read_timestamp', F.current_timestamp()).select("*","_metadata.file_name","_metadata.file_size")
print(df.count())
df.show(5)

In [0]:
display(df.limit(20))

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite") \
    .saveAsTable(bronze_table)



In [0]:
#move all files to processed folder: It is a best practice to move all the files from landing to processed after save data into bronze table 

# files = dbutils.fs.ls(landing_path)
# for file in files:
#   dbutils.fs.mv(file.path, f'{processed_path}/{file.name}',True)


In [0]:
df_orders = spark.sql(f'select * from {bronze_table}')

In [0]:
display(df_orders.limit(2))

In [0]:
df_orders = df_orders.filter(F.col("order_qty").isNotNull())

In [0]:
df_orders = df_orders.withColumn(
    "customer_id",F.when(F.col("customer_id").rlike(r'^[0-9]+$'), F.col("customer_id"))
    .otherwise("999999")
    .cast("bigint")                      
)

In [0]:
df_orders = df_orders.withColumnRenamed("order_placment_date", "date") \
    .withColumnRenamed("product_id", "product_code") \
    .withColumnRenamed("customer_id", "customer_code") \
    .withColumnRenamed("order_qty", "sold_quantity")


In [0]:
df_orders = df_orders.withColumnRenamed("order_placement_date", "date")
display(df_orders.limit(2))

In [0]:
df_orders = df_orders.withColumn(
    "date",F.regexp_replace(F.col("date"),r"^[A-Za-z]+,\s*","")
    )
df_orders = df_orders.withColumn("date",
F.coalesce(
    F.try_to_date("date","yyyy/MM/dd"),
    F.try_to_date("date","dd-MM-yyyy"),
    F.try_to_date("date","dd/MM/yyyy"),
    F.try_to_date("date","MMMM dd, yyyy"))
)

df_orders = df_orders.dropDuplicates(["order_id","date","customer_code","product_code","sold_quantity"])


In [0]:
null_counts = df_orders.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_orders.columns])
display(null_counts)

In [0]:
#product code is different it it the unique sha hash value
df_orders = df_orders.withColumnRenamed("product_code","product_id")

In [0]:
df_products = spark.table("fmcg.silver.products")
display(df_products)

In [0]:
df_joined = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])

In [0]:
display(df_joined.limit(20)) 


In [0]:
# save data to silver layer: when we first time update so we can directy write but best case is to check weather a same table present or check for conditions
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_code = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll()


In [0]:
print(silver_table)

### gold processing

In [0]:
df_gold = spark.sql(f'select order_id,date,customer_code,product_code,product_id,sold_quantity from {silver_table}')
display(df_gold)

In [0]:
gold_table = "fmcg.gold.sb_fact_orders"
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


In [0]:
%sql
drop table if exists fmcg.gold.orders;

### merge with parent company


In [0]:
df_child = spark.sql(f'select date,product_code,customer_code,sold_quantity from {gold_table}')
display(df_child)

In [0]:
df_monthly = (df_child
    #1. Get month start date (e.g., 2025-11-30 - 2025-11-01)
    .withColumn("month_start", F.trunc("date", "MM")) # or F.date_trunc("month", "date")

    #2.Group at monthly grain by month_start product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )
    .withColumnRenamed("month_start", "date")
)

In [0]:
display(df_monthly.limit(2))

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()